# Build and evaluate a ticket-routing prompt

An internal support queue routes tickets to **billing**, **technical** or **other**. In the lessons you wrote the pieces one at a time. Here you join them into a small pipeline and use it to decide between two prompt versions:

1. render a complete text prompt: instruction, allowed labels, fallback, examples and the new ticket;
2. write a fixture-driven mock model, so the pipeline runs with no model service;
3. route a set of tickets through prompt, mock and validator;
4. compare two prompt variants with a paired report;
5. run the comparison end to end and decide what it tells you.

Python standard library only. Fill in each `TODO` cell, then run the check cell after it. Before submitting, restart the kernel and run all cells.

In [ ]:
import json

LABELS = ["billing", "technical", "other"]
FALLBACK = "other"

# Demonstrations that go inside the prompt. They must never appear in the evaluation set.
EXAMPLES = [("Charged twice", "billing"), ("App will not start", "technical")]

# Two prompt variants. B adds an explicit boundary rule.
BASE = "Route the support ticket to one queue."
RULE = "Payment problems are billing; product faults are technical."
VARIANT_A = BASE
VARIANT_B = BASE + " " + RULE


def parse_label(text):
    """The validator from the lessons: return the label, or None for any invalid reply."""
    try:
        value = json.loads(text)
    except json.JSONDecodeError:
        return None
    if not isinstance(value, dict) or set(value) != {"label"}:
        return None
    label = value["label"]
    return label if label in LABELS else None


## 1. Render the full prompt

Write `render_prompt(instruction, labels, fallback, examples, ticket)`. Return one string whose lines are, in order:

1. the instruction;
2. `Allowed labels: ` followed by the labels joined with `", "`;
3. `If the ticket does not give enough evidence, answer <fallback>.`
4. `Reply with JSON only: {"label": "<one allowed label>"}` (literally this text);
5. an empty line;
6. for each example: `Example ticket: <ticket as a JSON string>`, then `Example label: <label>`, then an empty line;
7. finally `Ticket: <new ticket as a JSON string>`, with no label after it.

Use `json.dumps(text)` for ticket text, so quotes and line breaks cannot break the layout. Join the lines with `"\n"`.

Raise `ValueError` if the fallback is not one of the labels or an example label is not one of the labels: a prompt that contradicts its own contract should never reach a model.

For example, `render_prompt("Sort mail.", ["spam", "ham"], "ham", [("Win cash", "spam")], "Hi")` returns

```text
Sort mail.
Allowed labels: spam, ham
If the ticket does not give enough evidence, answer ham.
Reply with JSON only: {"label": "<one allowed label>"}

Example ticket: "Win cash"
Example label: spam

Ticket: "Hi"
```

In [ ]:
def render_prompt(instruction, labels, fallback, examples, ticket):
    # TODO
    raise NotImplementedError


In [ ]:
expected_mail = "\n".join([
    "Sort mail.",
    "Allowed labels: spam, ham",
    "If the ticket does not give enough evidence, answer ham.",
    'Reply with JSON only: {"label": "<one allowed label>"}',
    "",
    'Example ticket: "Win cash"',
    "Example label: spam",
    "",
    'Ticket: "Hi"',
])
got = render_prompt("Sort mail.", ["spam", "ham"], "ham", [("Win cash", "spam")], "Hi")
assert got == expected_mail, "Layout differs from the spec. Got:\n" + got

p = render_prompt(VARIANT_B, LABELS, FALLBACK, EXAMPLES, 'Says "hi"\nthen leaves')
assert p.startswith(VARIANT_B + "\n"), "The instruction comes first"
assert "Allowed labels: billing, technical, other" in p, "List every allowed label"
assert "answer other." in p, "State the fallback"
assert p.splitlines()[-1] == 'Ticket: "Says \\"hi\\"\\nthen leaves"', "Escape the new ticket with json.dumps and put it last: " + p.splitlines()[-1]
assert p.count("Example label:") == len(EXAMPLES), "Only the examples carry labels"

for bad, why in [((LABELS, "unsure", []), "a fallback outside the labels"),
                 ((LABELS, FALLBACK, [("Refund", "refunds")]), "an example label outside the labels")]:
    try:
        render_prompt(BASE, bad[0], bad[1], bad[2], "Help")
    except ValueError:
        pass
    else:
        raise AssertionError("Raise ValueError for " + why)
print("OK")


## 2. A fixture-driven mock model

Real model calls are slow, cost money and vary between runs. For developing the pipeline, replace the model with a **mock** that replays recorded replies.

`FIXTURES` maps each evaluation ticket to its expected label and the raw reply recorded for each prompt variant. Some replies are deliberately invalid, as real ones sometimes are.

Write `fake_model(prompt)`:

- read the ticket from the prompt's **last** line (strip `Ticket: ` and `json.loads` the rest; example lines must not confuse it);
- use the `"B"` reply if `RULE` appears in the prompt, otherwise the `"A"` reply;
- raise `KeyError` for a ticket that has no fixture. A mock that invents a reply hides gaps in your test data.

In [ ]:
FIXTURES = {
    "I was charged twice for March":                {"expected": "billing",   "A": '{"label": "billing"}',   "B": '{"label": "billing"}'},
    "The app crashes when I open settings":         {"expected": "technical", "A": '{"label": "technical"}', "B": '{"label": "technical"}'},
    "Checkout took my card but shows an error":     {"expected": "billing",   "A": '{"label": "technical"}', "B": '{"label": "billing"}'},
    "Login button does nothing":                    {"expected": "technical", "A": '{"label": "technical"}', "B": '{"label": "technical"}'},
    "please help":                                  {"expected": "other",     "A": '{"label": "billing"}',   "B": '{"label": "other"}'},
    "Refund for a duplicate order":                 {"expected": "billing",   "A": '{"label": "billing"}',   "B": '```json\n{"label": "billing"}\n```'},
    "Export to CSV produces an empty file":         {"expected": "technical", "A": '{"label": "technical"}', "B": '{"label": "technical"}'},
    "Invoice shows the wrong VAT number":           {"expected": "billing",   "A": '{"label": "other"}',     "B": '{"label": "billing"}'},
    "Where is your office?":                        {"expected": "other",     "A": '{"label": "other"}',     "B": '{"label": "other"}'},
    "Password reset email never arrives":           {"expected": "technical", "A": '{"label": "Technical"}', "B": '{"label": "billing"}'},
}
TICKETS = list(FIXTURES)
EXPECTED = [FIXTURES[t]["expected"] for t in TICKETS]
assert not set(TICKETS) & {t for t, _ in EXAMPLES}, "Demonstrations leaked into the evaluation set"


In [ ]:
def fake_model(prompt):
    # TODO
    raise NotImplementedError


In [ ]:
pa = render_prompt(VARIANT_A, LABELS, FALLBACK, EXAMPLES, "please help")
pb = render_prompt(VARIANT_B, LABELS, FALLBACK, EXAMPLES, "please help")
assert fake_model(pa) == '{"label": "billing"}', "Variant A replays the A fixture"
assert fake_model(pb) == '{"label": "other"}', "Variant B (contains RULE) replays the B fixture"
pc = render_prompt(VARIANT_A, LABELS, FALLBACK, EXAMPLES, "Invoice shows the wrong VAT number")
assert fake_model(pc) == '{"label": "other"}', "Read the ticket from the last line, not from an example line"
try:
    fake_model(render_prompt(VARIANT_A, LABELS, FALLBACK, EXAMPLES, "A ticket with no fixture"))
except KeyError:
    pass
else:
    raise AssertionError("Raise KeyError for a ticket with no recorded reply")
print("OK")


## 3. Route a set of tickets

Write `route(tickets, instruction)`. For each ticket, render the prompt (with `LABELS`, `FALLBACK` and `EXAMPLES`), call `fake_model`, and validate the reply with `parse_label`. Return the list of validated labels, with `None` wherever the reply was invalid. Never replace `None` with a guess: an invalid reply is a failure, while `other` is a real decision.

In [ ]:
def route(tickets, instruction):
    # TODO
    raise NotImplementedError


In [ ]:
got_a = route(TICKETS, VARIANT_A)
got_b = route(TICKETS, VARIANT_B)
assert got_a == ["billing", "technical", "technical", "technical", "billing", "billing", "technical", "other", "other", None], f"Variant A labels: {got_a}"
assert got_b == ["billing", "technical", "billing", "technical", "other", None, "technical", "billing", "other", "billing"], f"Variant B labels: {got_b}"
assert route([], VARIANT_A) == [], "No tickets, no labels"
print("invalid replies: A", got_a.count(None), "B", got_b.count(None))
print("OK")


## 4. A paired report

Write `paired_report(expected, labels_a, labels_b)`. Treat `None` (an invalid reply) as wrong. Return a dict with

- `"gains"`: A wrong, B right;
- `"regressions"`: A right, B wrong;
- `"both_right"` and `"both_wrong"`;
- `"discordant"`: gains plus regressions, the only cases that tell the variants apart.

Raise `ValueError` if the three lists differ in length: misaligned predictions would silently compare the wrong cases.

In [ ]:
def paired_report(expected, labels_a, labels_b):
    # TODO
    raise NotImplementedError


In [ ]:
r = paired_report(["a", "b", "c", "d"], ["a", "x", "c", None], ["a", "b", "y", "z"])
assert r == {"gains": 1, "regressions": 1, "both_right": 1, "both_wrong": 1, "discordant": 2}, f"Mixed case: {r}"
r = paired_report(["a", "b"], [None, None], ["a", "b"])
assert r == {"gains": 2, "regressions": 0, "both_right": 0, "both_wrong": 0, "discordant": 2}, f"None must count as wrong: {r}"
r = paired_report([], [], [])
assert r == {"gains": 0, "regressions": 0, "both_right": 0, "both_wrong": 0, "discordant": 0}, f"Empty input: {r}"
for args in ((["a"], [], ["a"]), (["a", "b"], ["a", "b"], ["a"]), ([], ["a"], ["a"])):
    try:
        paired_report(*args)
    except ValueError:
        pass
    else:
        raise AssertionError(f"Raise ValueError for mismatched lengths {tuple(map(len, args))}")
print("OK")


## 5. Compare the two prompt variants

Run both variants end to end and build the report. Nothing new to write: this cell checks that the pieces fit together.

In [ ]:
labels_a = route(TICKETS, VARIANT_A)
labels_b = route(TICKETS, VARIANT_B)
report = paired_report(EXPECTED, labels_a, labels_b)
acc_a = sum(a == g for a, g in zip(labels_a, EXPECTED)) / len(EXPECTED)
acc_b = sum(b == g for b, g in zip(labels_b, EXPECTED)) / len(EXPECTED)
print("accuracy A", acc_a, "B", acc_b)
print(report)
for t, g, a, b in zip(TICKETS, EXPECTED, labels_a, labels_b):
    if (a == g) != (b == g):
        print(f"  {'gain' if b == g else 'regression':10} {t!r}: expected {g}, A={a}, B={b}")
assert report == {"gains": 3, "regressions": 1, "both_right": 5, "both_wrong": 1, "discordant": 4}, f"Report: {report}"
assert (acc_a, acc_b) == (0.6, 0.8), f"Accuracies: {acc_a}, {acc_b}"
print("OK")


## Interpretation

Answer in this cell.

1. B wins 3 to 1 on the discordant cases. If the two variants were equally good, each discordant case would be a coin flip. How likely is a split at least this favourable to B by chance? Is that enough to adopt B?
2. B's one regression is an invalid reply, not a wrong label. What would you change, and why should that change not be a silent repair inside `parse_label`?
3. Write three challenge tickets for a fresh held-out set, including one ambiguous case, with the label your rubric assigns each.
4. State the result (number of cases, gains, regressions) that would make you adopt B, and what you would ask a human reviewer to check.

**Your answer**
